# Gestionnaires de contexte

## Implémentation d'un gestionnaire de contexte avec les méthodes magiques

Créez une classe `StdoutRedirector` qui sera utilisable comme un gestionnaire de contexte. Vous pouvez consulter la documentation du type [`ContextManager`](https://docs.python.org/fr/3/library/stdtypes.html#typecontextmanager). Cette classe stockera la valeur de `sys.stdout` à l'entrée dans le contexte, et la remplacera par l'argument qui lui est donné à sa création. À la sortie du contexte, l'ancienne sortie standard devra être restaurée.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import io
import sys


class StdoutRedirector:
  def __init__(self, new_stdout):
    self.new_stdout = new_stdout

  def __enter__(self):
    self.old_stdout = sys.stdout
    sys.stdout = self.new_stdout

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = self.old_stdout
    return False


with io.StringIO() as string_io:
  with StdoutRedirector(string_io):
    print("Hello World")
  print(f"Valeur de la StringIO : {string_io.getvalue()}")

## Implémentation d'un gestionnaire de contexte avec un générateur

Recréez le contexte StdoutRedirector mais cette fois à l'aide de la méthode [`contextlib.contextmanager`](https://docs.python.org/fr/3/library/contextlib.html#contextlib.contextmanager), en utilisant un générateur.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import contextlib
import io
import sys


@contextlib.contextmanager
def StdoutRedirector(new_stdout):
  try:
    old_stdout = sys.stdout
    sys.stdout = new_stdout
    yield
  finally:
    sys.stdout = old_stdout


with io.StringIO() as string_io:
  with StdoutRedirector(string_io):
    print("Hello World")
  print(f"Valeur de la StringIO : {string_io.getvalue()}")

## Implémentation d'un contexte qui supprime un type d'exception

Implémentez avec `contextlib.contextmanager` un gestionnaire de contexte qui supprime un type d'exception (en stoppe la propagation) dans le corps du contexte.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Iterator
import contextlib


@contextlib.contextmanager
def suppress(exc_type: type[Exception]) -> Iterator[None]:
  try:
    yield
  except exc_type:
    pass


with suppress(NameError):
  print(abcd)

## Réimplémentation du décorateur `contextlib.contextmanager` (difficile)

À l'aide d'un décorateur, qui créera une classe avec les méthodes `__init__`, `__enter__` et `__exit__`, réimplémentez le décorateur [`contextlib.contextmanager`](https://docs.python.org/fr/3/library/contextlib.html#contextlib.contextmanager).

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Callable, Iterator
from types import TracebackType
from typing import ContextManager
import io

def contextmanager[**P, T](generator: Callable[P, Iterator[T]]
                           ) -> Callable[P, ContextManager[T]]:
  class Wrapper:
    def __init__(self, *args: P.args, **kwargs: P.kwargs) -> None:
      self._generator = generator(*args, **kwargs)

    def __enter__(self) -> T:
      return next(self._generator)

    def __exit__[U: BaseException](self,
                                   exc_type: type[U] | None,
                                   exc_val: U | None,
                                   exc_tb: TracebackType | None) -> bool:
      try:
        if exc_type is None:
          next(self._generator)
        else:
          self._generator.throw(exc_val)
      except StopIteration:
        pass
      else:
        raise RuntimeError("Generator used as a context should yield only once")
      return True

  return Wrapper

@contextmanager
def suppress(exc_type):
  try:
    yield
  except exc_type:
    pass

with suppress(NameError):
  print(abcd)

with suppress(NameError):
  1 / 0